In [ ]:
import pandas as pd
from spswarehouse.warehouse import Warehouse
from spswarehouse.table_utils import *
from spswarehouse.googlesheets import GoogleSheets

In [ ]:
# This is the example from https://developers.google.com/sheets/api/quickstart/python
# Replace it with your own spreadsheet
spreadsheet_id = '1BxiMVs0XRA5nFMdKvBdBZjgmUUqptlbs74OgvE2upms'
spreadsheet = GoogleSheets.open_by_key(spreadsheet_id)

# GoogleSheet
sheet = spreadsheet.sheet1

# Sheet as a DataFrame
df = pd.DataFrame(sheet.get_all_records())

# As a CSV file
csv_filename = '~/Desktop/tmp.csv'
df.to_csv(csv_filename, index=False)

In [ ]:
# Get the "CREATE TABLE" statement
table_name = sanitize_string('test-{id}'.format(id=spreadsheet_id)).upper()
schema_name = 'wild_west'

# You can specify either a sheet, dataframe, or CSV file as your data source. 
# Note that create_table_stmt tries to guess the column type based on the data
sql = create_table_stmt(table_name, schema=schema_name, google_sheet=sheet)
sql_df = create_table_stmt(table_name, schema=schema_name, dataframe=df)
sql_csv = create_table_stmt(table_name, schema=schema_name, csv_filename=csv_filename)
# The above are equivalent because the data is the same

print(sql)

In [ ]:
# Create the table in the Snowflake warehouse
results = Warehouse.execute(sql)
print(results)

In [ ]:
# Like create_table_stmt, you can upload from a sheet, dataframe, or CSV file
Warehouse.upload_google_sheet(table_name, schema_name,  google_sheet=sheet)
#Warehouse.upload_local_csv(table_name, schema_name,  csv_filename=csv_filename)
#Warehouse.upload_df(table_name, schema_name,  dataframe=df)
#Warehouse.upload_google_drive_csv(table_name, schema_name,  google_drive_id=google_drive_id) #See googledrive-example.ipynb for more details

# Additional parameters that you can pass to both upload_to_warehouse and create_table_stmt
# encoding='<encoding-name>': default value is 'utf-8'
# force_string=True: default value is False. This forces all columns to be interpreted as a string (good for leading zeros)
#     force_string does NOT do anything if you pass a dataframe
# batch_size=<integer>: upload_to_warehouse only; default value is 200


In [ ]:
# Read the data back
Warehouse.read_sql('SELECT * FROM {schema}.{table} LIMIT {limit}'.format(schema=schema_name, table=table_name, limit=30))

In [ ]:
# Delete the test table
Warehouse.execute(f'DROP TABLE {schema_name}.{table_name}')